# MODELO_DIMENSIONAL_SQL — RetailNova

Componente SQL del Proyecto Final · Data Lab  
Caso #1 · RetailNova — Plataforma Analítica en Microsoft Fabric  
Capa: `02_TRANS/MODELO_DIMENSIONAL_SQL/`

---

## Descripción del Dataset

**Fuente:** [Retail Sales Dataset — Kaggle](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset)  
**Tabla de origen (Bronze):** `Bronze.dbo.ventas_raw`

El dataset contiene transacciones de ventas de una cadena retail con los siguientes campos:

| Columna | Tipo | Descripción |
|---|---|---|
| `transaction_id` | String | Identificador único de transacción (PK) |
| `date` | Date | Fecha de la venta |
| `customer_id` | String | Identificador del cliente |
| `gender` | String | Género del cliente |
| `age` | Integer | Edad del cliente |
| `product_category` | String | Categoría del producto vendido |
| `quantity` | Integer | Unidades vendidas |
| `price_per_unit` | Double | Precio unitario del producto |
| `total_amount` | Double | Ingreso total de la transacción |

---

## Arquitectura y Decisiones de Diseño

El componente SQL sigue la arquitectura **Medallón** operando sobre las capas **Silver** y **Gold**:

```
Bronze (ventas_raw)
    │
    ▼
Silver (limpieza y calidad de datos)
    │
    ▼
Gold (modelo estrella: dims + hechos + vistas)
```

### Decisiones técnicas clave

- Las **claves surrogadas** de todas las dimensiones se generan con `ROW_NUMBER() OVER (ORDER BY ...)`, desacoplando el modelo del dato fuente.
- Todos los objetos incluyen `DROP TABLE/VIEW IF EXISTS` para garantizar **idempotencia** (re-ejecutabilidad sin errores).
- El dataset original de Kaggle no contiene errores de calidad, por lo que se **simulan problemas reales** (nulos e inconsistencias de capitalización) en `limpieza_sql` para demostrar las técnicas de limpieza.
- No se usa `SELECT *` en ninguna consulta de entrega final.
- Todo el código sigue convención **snake_case** sin tildes, mayúsculas ni caracteres especiales en nombres de objetos.

---

## Modelo Dimensional en Estrella

```
                  ┌─────────────┐
                  │  dim_fecha  │
                  │─────────────│
                  │ id_fecha PK │
                  │ fecha       │
                  │ anio        │
                  │ mes         │
                  │ dia         │
                  └──────┬──────┘
                         │
┌──────────────┐   ┌─────▼──────────┐   ┌─────────────────┐
│ dim_categoria│   │  Hechos_Ventas │   │   dim_genero    │
│──────────────│   │────────────────│   │─────────────────│
│ id_categoria │◄──│ id_categoria FK│   │ id_genero    PK │
│ product_cat. │   │ id_fecha     FK│──►│ nombre_genero   │
└──────────────┘   │ id_genero    FK│   └─────────────────┘
                   │ transaction_id │
                   │ customer_id    │
                   │ quantity       │
                   │ price_per_unit │
                   │ total_amount   │
                   │ age            │
                   └────────────────┘
```

### Tablas del modelo

| Objeto | Tipo | Descripción |
|---|---|---|
| `dim_categoria` | Dimensión | Categorías únicas de producto con clave surrogada |
| `dim_fecha` | Dimensión | Calendario con año, mes y día derivados de las fechas de transacción |
| `dim_genero` | Dimensión | Géneros únicos de clientes normalizados tras la limpieza |
| `Hechos_Ventas` | Tabla de hechos | Transacciones con métricas de venta y FK a las tres dimensiones |

---

## Estructura de Archivos

```
MODELO_DIMENSIONAL_SQL/
├── limpieza_sql.ipynb    # Diagnóstico y limpieza de datos (Bronze → Silver)
├── modelado_sql.ipynb    # DDL del esquema estrella (Silver → Gold)
├── analisis_sql.ipynb    # 5 consultas analíticas con CTEs y Window Functions
├── vistas_sql.ipynb      # 3 vistas para consumo por visualización y modelo semántico
└── README.md             # Este archivo
```

---

## Descripción de Notebooks

### `limpieza_sql.ipynb` — Diagnóstico y Calidad de Datos

Opera sobre `Bronze.dbo.ventas_raw` y produce la tabla `silver` lista para modelado.

**Pasos ejecutados:**

1. **Carga desde Bronze** — lectura de `ventas_raw` y registro como vista temporal `silver_1`.
2. **Simulación de datos sucios** — se inyectan errores controlados en `silver_sucio` para demostrar técnicas de limpieza:
   - `gender`: 5% nulos + inconsistencias de capitalización (UPPER/lower)
   - `age`: 3% nulos
   - `total_amount`: 15% nulos
3. **Diagnóstico previo a la limpieza:**
   - Conteo total de registros
   - Nulos por columna (todos los campos)
   - Duplicados en clave primaria `transaction_id`
4. **Limpieza y creación de tabla `silver`:**
   - Nulos de `gender` tratados con `COALESCE` + `INITCAP` para normalizar capitalización
   - Nulos de `age` imputados con la mediana del género correspondiente usando `PERCENTILE_CONT`
   - Nulos de `total_amount` recalculados desde `quantity * price_per_unit`
   - Deduplicación con `ROW_NUMBER()` sobre `transaction_id`

**Criterio de aceptación:** 0 nulos sin tratar · 0 duplicados en PK · script re-ejecutable.

---

### `modelado_sql.ipynb` — Esquema Estrella

Opera sobre `silver` y construye el modelo dimensional en Gold.

**Objetos creados:**

| Objeto | Lógica de creación |
|---|---|
| `dim_categoria` | `SELECT DISTINCT product_category` + `ROW_NUMBER()` como `id_categoria` |
| `dim_fecha` | `SELECT DISTINCT date` + extracción de `YEAR`, `MONTH`, `DAY` |
| `dim_genero` | `SELECT DISTINCT gender` + `ROW_NUMBER()` como `id_genero` |
| `Hechos_Ventas` | `JOIN` de `silver` con las tres dimensiones para resolver FK |

Cada objeto incluye `DROP TABLE IF EXISTS` previo para garantizar idempotencia.

---

### `analisis_sql.ipynb` — Consultas Analíticas

5 consultas que responden las preguntas de negocio de RetailNova usando CTEs encadenadas y Window Functions.

| # | Pregunta de negocio | Window Functions usadas |
|---|---|---|
| 1 | ¿Cuáles son las categorías con mayor venta? | `RANK() OVER (ORDER BY)`, `SUM() OVER ()` para % de participación |
| 2 | ¿Qué clientes generan más ingresos? | `DENSE_RANK() OVER (ORDER BY)`, `NTILE(4)` para segmentación en cuartiles |
| 3 | ¿Cómo evolucionan las ventas en el tiempo? | `SUM() OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`, `LAG()` para variación vs mes anterior |
| 4 | ¿Qué productos tienen baja rotación? | `PERCENT_RANK() OVER (ORDER BY)`, `SUM() OVER ()` para % de participación |
| 5 | ¿Cómo se comportan los géneros por categoría? | `AVG() OVER (PARTITION BY gender)`, `AVG() OVER (PARTITION BY product_category)`, `AVG() OVER ()` |

**Segmentación de clientes (Consulta 2):**

| Cuartil | Segmento |
|---|---|
| 1 | Premium — top 25% por gasto |
| 2 | Alto — entre 25% y 50% |
| 3 | Medio — entre 50% y 75% |
| 4 | Bajo — último 25% |

**Alertas de rotación (Consulta 4):**

| Rango de percentil | Alerta |
|---|---|
| < 0.25 | 🔴 Baja rotación |
| 0.25 – 0.75 | 🟡 Rotación media |
| > 0.75 | 🟢 Alta rotación |

---

### `vistas_sql.ipynb` — Vistas para Consumo Analítico

3 vistas construidas sobre el modelo estrella, diseñadas para ser consumidas por el dashboard y el modelo semántico en Gold.

| Vista | Descripción | Columnas principales |
|---|---|---|
| `vista_ventas_por_categoria` | Ingresos, unidades y transacciones por categoría, ordenadas de mayor a menor ingreso | `categoria`, `total_transacciones`, `total_unidades`, `total_ingresos` |
| `vista_ventas_mensuales` | Evolución mensual de ventas con período en formato `yyyy-MM` | `anio_mes`, `total_transacciones`, `total_unidades`, `total_ingresos` |
| `vista_ventas_por_genero` | Comparativo de ventas por género del cliente | `genero`, `total_transacciones`, `total_unidades`, `total_ingresos` |

Cada vista incluye `DROP VIEW IF EXISTS` previo y su prueba de ejecución `SELECT * FROM <vista>`.

---

## Reglas de Gobernanza Aplicadas

- Convención **snake_case** en todos los objetos, columnas y variables
- `DROP IF EXISTS` antes de cada `CREATE` para garantizar re-ejecutabilidad
- Sin `SELECT *` en consultas de entrega final
- Sin valores mágicos sin comentario explicativo
- Cada bloque SQL tiene encabezado con propósito, técnica usada y lógica de la CTE
- Scripts probados con ejecución exitosa en Microsoft Fabric (SparkSQL)

---

## Preguntas de Negocio Respondidas

| Pregunta | Notebook | Consulta / Vista |
|---|---|---|
| ¿Cuáles son las categorías con mayor venta? | `analisis_sql` + `vistas_sql` | Consulta 1 · `vista_ventas_por_categoria` |
| ¿Qué clientes generan más ingresos? | `analisis_sql` | Consulta 2 |
| ¿Cómo evolucionan las ventas en el tiempo? | `analisis_sql` + `vistas_sql` | Consulta 3 · `vista_ventas_mensuales` |
| ¿Qué productos tienen baja rotación? | `analisis_sql` | Consulta 4 |
| ¿Cómo se comportan los géneros por categoría? | `analisis_sql` + `vistas_sql` | Consulta 5 · `vista_ventas_por_genero` |
